In [ ]:
/*
================================================================================
  DEDUPLICATING DATA — EXPLAINED LIKE WE'RE JUST TALKING ABOUT IT
================================================================================

  Real scenarios that all boil down to the SAME underlying problem:

    - Your wifi drops mid-upload, so you hit "upload" 5 times just to be
      safe. Now the same 50MB video is sitting on the server FIVE times.
    - A message queue isn't sure if a message got processed, so it sends
      it again "just in case." Now your customer got charged TWICE.
    - Two computers both sync the same record from different places, and
      now your database has 3 "different" copies of the same person.

  All THREE of these are the exact same root problem: DUPLICATE DATA.

  Here's the thing that surprises a lot of beginners: in real, distributed
  systems (multiple computers talking to each other over networks),
  duplicates are NOT a bug you can fully eliminate. They're a NORMAL,
  EXPECTED side effect of how networks and retries work. Networks drop
  stuff. Computers retry. People double-click buttons. It just happens.

  So the real interview question isn't:
      "How do I make sure duplicates NEVER happen?"
  It's:
      "Duplicates WILL happen — so where might they show up, and how do
       I make sure it's SAFE when they do?"


--------------------------------------------------------------------------------
  1. WHY DO DUPLICATES EVEN HAPPEN? (it's not sloppy code, it's physics)
--------------------------------------------------------------------------------

  ANALOGY: Imagine texting a friend "did you eat yet?" You don't get a
  reply for 10 minutes, so you send it again, thinking your first text
  didn't go through. But actually — it DID go through, your friend just
  hasn't replied yet. Now they got the same question twice. Was your
  phone "broken"? No! You just couldn't tell if your message worked, so
  you played it safe and sent it again.

  THE NETWORK'S DIRTY SECRET:
    When a computer sends a request, it basically has 2 choices:
      (a) "At-most-once": send it, and NEVER retry, even if you're not
          sure it worked (risk: you might lose real data)
      (b) "At-least-once": retry until you get confirmation it worked
          (risk: you might process the same thing twice)

    Almost every real system picks option (b), because losing data is
    usually worse than occasionally processing something twice. But
    that means duplicates become the "cost of doing business."

    Here's the classic scenario:

        Client                          Server
          │──Request (ID: 123)─────────▶│
          │                              │──Process request
          │◀────Response LOST in transit─│  (actually succeeded!)
          │
          │  [Client has NO IDEA if it worked]
          │
          │──Retry SAME request (123)──▶│
          │                              │──Processes it AGAIN 😬

    The client genuinely CANNOT tell the difference between "my request
    never arrived" and "it worked, but the reply got lost on the way
    back." So the safe move is: just retry. Which sometimes means doing
    the same thing twice.

  OTHER SOURCES OF DUPLICATES IN DISTRIBUTED SYSTEMS:
    - Retries: load balancers, queues, and services all auto-retry
      failed calls, often without knowing other layers already retried too
    - Replication: the same data copied across machines can arrive via
      multiple paths
    - Event publishing: a service recovering from a crash might
      re-publish an event it already sent
    - Syncing data: merging records from different sources often creates
      overlap

  AND OF COURSE — HUMANS CAUSE DUPLICATES TOO:
    Double-clicking "submit," refreshing mid-form-submission, uploading
    the same file from 2 different devices, re-running something they
    THINK failed. Any system that deals with real users has to expect this.


--------------------------------------------------------------------------------
  2. NOT ALL DUPLICATES ARE THE SAME — 4 DIFFERENT "TYPES"
--------------------------------------------------------------------------------

  This is the big "aha" of this whole topic: the fix depends on WHAT
  kind of duplicate you're dealing with. There are 4 main flavors:

    Type      | Goal                              | Example
    ------------|-------------------------------------|--------------------------------
    Content     | Store the same FILE only once        | Same photo uploaded by 1,000 people
    Message     | Make repeated DELIVERIES harmless     | A queue retries the same message
    Record      | Prevent duplicate DATABASE ROWS       | Only one account per email address
    Request     | Make API CALLS safely repeatable      | Retry a payment without double-charging

  Each one needs a genuinely different tool. Let's go through them one at
  a time.


--------------------------------------------------------------------------------
  3. STRATEGY 1 — CONTENT-BASED DEDUPLICATION (hashing files)
--------------------------------------------------------------------------------

  ANALOGY: Imagine a library where, instead of asking "what's this book
  called?", the librarian instead FINGERPRINTS every single book by its
  actual CONTENT. Two books with the exact same text — no matter what
  title is printed on the cover — get recognized as "the same book" and
  only ONE physical copy needs to sit on the shelf.

  HOW IT WORKS:
    You run the file's bytes through a hashing function (SHA-256 is the
    standard choice). The cool property of a good hash function: the
    EXACT same content always produces the EXACT same hash, no matter
    the filename, who uploaded it, or when.

        User A uploads "photo.jpg"      ─┐
        User B uploads "vacation.jpg"    ├─▶ SHA-256 ──▶ same hash: a3f2b7c...
        User C uploads "image.jpg"      ─┘        (all 3 files are byte-identical)

                                    ▼
                          Object Store: stored ONLY ONCE

    UPLOAD FLOW:
      1. Compute the SHA-256 hash of the file
      2. Check: does this hash already exist in our metadata database?
      3. If YES → just create a reference/pointer to the existing file
         (skip the actual upload — no new storage needed!)
      4. If NO → upload it for real, and remember its hash

    The savings can be huge: if 1,000 people upload the exact same
    50MB vacation photo, you store 50MB total instead of 50GB. Viral
    memes, default profile pictures, common stock images — all stored
    ONCE no matter how many "owners" they have.

  3.1 CHUNK-LEVEL DEDUPLICATION (for files that are ALMOST the same)

    File-level hashing only helps for files that are 100% identical.
    But what about two versions of the same document where you only
    changed one paragraph? They're 99% the same, but a whole-file hash
    would treat them as completely different files.

    ANALOGY: Instead of fingerprinting the WHOLE book, fingerprint each
    individual PAGE. If you only edited page 12, pages 1-11 and 13-20
    are still recognized as identical to before — only page 12 needs to
    be re-stored.

        File A (original):  [Chunk1] [Chunk2] [Chunk3] [Chunk4]
        File B (edited):    [Chunk1] [ NEW  ] [Chunk3] [Chunk4]
                                ↑        ↑        ↑        ↑
                              same    DIFFERENT  same     same

        → Only "NEW" chunk needs to be stored. The other 3 are reused!

    HOW DO YOU DECIDE WHERE TO "CUT" A FILE INTO CHUNKS?

      Approach          | How it works                          | Best for
      --------------------|------------------------------------------|---------------------------
      Fixed-size          | Cut every N bytes (like every 4MB)        | Simple to build
      Content-defined     | Use a "rolling hash" to find natural       | Better dedup when data
                          |  breakpoints based on the CONTENT itself   |  shifts around

      THE PROBLEM WITH FIXED-SIZE CHUNKING: if you insert just ONE byte
      at the very beginning of a file, EVERY chunk boundary after that
      point shifts over by one byte — suddenly ALL your chunks look
      "different," even though 99% of the content didn't actually change!

      Content-defined chunking fixes this by picking chunk boundaries
      based on PATTERNS in the data itself (not fixed positions). So
      inserting a byte at the start barely affects anything — most
      chunk boundaries stay right where they were.

  3.2 PROS AND CONS OF CONTENT-BASED DEDUP

    Pros                                | Cons
    ---------------------------------------|-----------------------------------------
    Big storage savings                   | Uses CPU to compute all those hashes
    Works across DIFFERENT users/systems  | Chunking adds real implementation complexity
    "Content-addressable" — nice property | Tiny (near-zero) risk of hash collisions
    Doubles as integrity verification      | ENCRYPTION BREAKS THIS — see below!

    IMPORTANT GOTCHA: if a file is encrypted BEFORE it's hashed, two
    identical files encrypted differently will have completely
    different hashes — so deduplication just... stops working. This is
    a REAL trade-off real companies had to make: Dropbox, for example,
    had to choose between strong client-side encryption and strong
    deduplication. You mostly can't have both.


--------------------------------------------------------------------------------
  4. STRATEGY 2 — MESSAGE DEDUPLICATION (for queues, retries, events)
--------------------------------------------------------------------------------

  ANALOGY: Imagine a restaurant kitchen where orders come in on paper
  tickets. If a ticket gets smudged and the waiter isn't sure the
  kitchen saw it, they hand in ANOTHER copy just in case. The kitchen
  needs a system to say "wait, I already made this exact order" so they
  don't cook the same meal twice.

  Most message queues default to "at-least-once" delivery — when in
  doubt, they'd rather deliver a message TWICE than risk losing it
  forever. That means as the CONSUMER (the thing processing messages),
  YOU need to handle the possibility of seeing the same message more
  than once.

  4.1 HOW IT WORKS

        Producer      Queue        Consumer         Dedup Store (Redis)
           │            │              │                    │
           │──msg_123──▶│              │                    │
           │            │──deliver────▶│                    │
           │            │              │──claim ID 123 atomically──▶│
           │            │              │◀────claim WON──────────────│
           │            │              │──process the message──     │
           │            │              │──mark completed (TTL 24h)─▶│
           │            │◀──acknowledge│                    │
           │            │              │
           │        [network hiccup — queue isn't sure it was acked]
           │            │              │
           │            │──deliver AGAIN (same msg_123)────▶│
           │            │              │──claim ID 123 atomically──▶│
           │            │              │◀── already completed! ─────│
           │            │◀──acknowledge (SKIP processing)───│

    THE FLOW IN PLAIN ENGLISH:
      1. Get a message with a stable, unique ID
      2. Try to "claim" that ID in a fast store like Redis
      3. Claim failed? Someone already processed (or is processing) it
         — skip or wait, depending on the situation
      4. Claim succeeded? Go ahead and process it for real
      5. Mark it as "completed" once done
      6. Only THEN tell the queue "yep, got it, don't resend"

    THE TTL (expiration time) MATTERS: without one, your "have I seen
    this?" store just grows forever. 24 hours is a common default, but
    payments/bookings might need to remember longer.

  4.2 THE SNEAKY RACE CONDITION

    A naive "check, then act" approach has a bug hiding in it. If TWO
    consumers grab the same message at almost the exact same instant:

        Consumer 1          Redis          Consumer 2
           │──check msg_123─▶│               │
           │◀────not found───│               │
           │                 │◀──check msg_123│
           │                 │────not found──▶│
           │──PROCESS (duplicate!)             │
           │                                   │──PROCESS (duplicate!)

    Both consumers checked, both saw "nothing there yet," and BOTH went
    ahead and processed. Oops.

    THE FIX: combine "check if it exists" and "claim it" into ONE
    ATOMIC step (not two separate steps with a gap in between). In
    Redis terms, that's a "SET if not exists, with an expiry" done as a
    single command — never two separate commands. Only ONE consumer can
    win that atomic race; everyone else immediately sees it's already
    taken.

  4.3 WHAT TO DO WHEN THINGS GO WRONG

    Scenario                  | What should happen
    -----------------------------|--------------------------------------
    Processing SUCCEEDS          | Keep the "claimed" key (blocks reprocessing)
    Processing FAILS             | Delete the key (let it retry)
    Consumer CRASHES mid-work    | Key auto-expires via TTL (let it retry)

    That TTL is actually doing double duty here — it's not just cleanup,
    it's also your SAFETY NET for crashed workers. If a worker claims a
    message and then dies before finishing, that claim eventually
    expires so someone else can pick it up.

  This approach gives you what's called "effectively-once" processing:
  the MESSAGE might arrive more than once, but the real-world SIDE
  EFFECT (charging a card, sending an email) only actually happens once.


--------------------------------------------------------------------------------
  5. STRATEGY 3 — DATABASE-LEVEL DEDUPLICATION (let the DB do the work)
--------------------------------------------------------------------------------

  ANALOGY: Instead of a bouncer at the door checking a guest list by
  hand (which can get confused if two people walk up at the exact same
  second), you install an automatic turnstile that PHYSICALLY only lets
  one person with a given ticket number through, period, no exceptions,
  guaranteed by the mechanism itself.

  5.1 UNIQUE CONSTRAINTS — often the simplest fix that exists

        users table:
          UNIQUE constraint on `email`

        Insert #1: email = john@example.com  →  ✅ Accepted
        Insert #2: email = john@example.com  →  ❌ REJECTED (duplicate key error)

    Constraint type    | Example                              | Use case
    ---------------------|-----------------------------------------|--------------------------------
    Single column         | UNIQUE(email)                         | One account per email
    Composite (multi-col) | UNIQUE(event_id, user_id)              | One registration per user, per event
    Partial               | UNIQUE(email) WHERE deleted = false    | Allow re-registering after a soft delete

  5.2 THE "CHECK-THEN-INSERT" TRAP (and the fix: UPSERT)

    A really common beginner mistake: "first check if it exists, THEN
    insert if it doesn't." This has the SAME race condition problem as
    section 4.2 — two requests can both check, both find nothing, and
    both try to insert at the same moment.

    The fix is an "UPSERT" — a single atomic database operation that
    means "insert this, but if it already exists, just update it
    instead," done as ONE indivisible step:

        -- PostgreSQL
        INSERT INTO users (email, name)
        VALUES ('john@example.com', 'John Doe')
        ON CONFLICT (email) DO UPDATE SET name = EXCLUDED.name;

        -- MySQL
        INSERT INTO users (email, name)
        VALUES ('john@example.com', 'John Doe')
        ON DUPLICATE KEY UPDATE name = VALUES(name);

    No separate "check" step, no race condition, no gap in time where
    two things can sneak through at once.

  5.3 FUZZY DEDUPLICATION — when duplicates AREN'T exact matches

    Real-world data is messy. "John Doe" and "JOHN DOE" are clearly the
    same person. "john@gmail.com" and "john.doe@gmail.com" might ALSO
    be the same person (Gmail quietly ignores dots in addresses!).
    Exact string matching misses these "obvious to a human" duplicates.

        New record: "John Doe", john@gmail.com
                          │
                          ▼
                Find similar records
                          │
              ┌───────────┼───────────────┐
              ▼           ▼               ▼
         Exact match  Fuzzy match    Multiple ambiguous
         (confident)  (confident)         matches
              │           │               │
              ▼           ▼               ▼
         Merge with    Merge with     Flag for a HUMAN
         existing      existing       to review

    TECHNIQUES FOR FUZZY MATCHING:
      - Normalization: lowercase everything, trim spaces, strip weird
        characters BEFORE comparing
      - Similarity functions: things like trigram similarity or edit
        distance ("Levenshtein distance") to measure "how close" two
        strings are
      - Phonetic matching: algorithms like Soundex that catch names
        that SOUND alike but are spelled differently ("Smith" / "Smyth")

    THE IMPORTANT LESSON HERE: when you're not confident, DON'T auto-merge.
    Flag it for a human instead. Accidentally creating a small duplicate
    is usually way less harmful than accidentally merging two DIFFERENT
    people's records together. When in doubt, don't guess — ask.


--------------------------------------------------------------------------------
  6. STRATEGY 4 — IDEMPOTENCY KEYS (for APIs — payments especially!)
--------------------------------------------------------------------------------

  "Idempotent" is a fancy word for a simple idea: doing something once
  has the SAME EFFECT as doing it many times. Flipping a light switch
  ON is NOT idempotent (flip it twice, it's back off). But "make sure
  the light is ON" IS idempotent (do it 100 times, same end result).

  ANALOGY: Imagine you hand a cashier a receipt number along with your
  payment and say "if you ever see this EXACT receipt number again,
  don't charge me again — just tell me the SAME result as before."
  That receipt number is the "idempotency key."

  This flips who's responsible: instead of the SERVER trying to guess
  whether something is a duplicate, the CLIENT explicitly says "hey,
  this is a retry of that other thing I sent you."

  6.1 HOW IT WORKS

        Client                Server              Redis            PaymentService
          │──POST /payments────▶│                    │                     │
          │  Idempotency-Key:   │──check key abc123─▶│                     │
          │  abc123             │◀────not found───────│                     │
          │                     │──store: "processing"▶│                    │
          │                     │──process payment───────────────────────▶│
          │                     │◀───success, txn_id: 789──────────────────│
          │                     │──store: {result}────▶│                    │
          │◀──200 OK, txn_id 789─│                    │                     │
          │
          │   [network drops before client even sees the response]
          │
          │──POST /payments (RETRY, same key)──────▶│                     │
          │  Idempotency-Key: abc123                 │──check key abc123──▶│
          │                                          │◀── found! completed ─│
          │◀──200 OK, txn_id: 789 (CACHED response)──│                     │

    The client gets the EXACT SAME response whether the request
    actually ran once, or was retried ten times. That means retries are
    now genuinely SAFE — they don't create new side effects.

  6.2 THE CLIENT'S PART OF THE DEAL

    The client has to follow 3 rules for this whole system to work:
      1. Generate a unique key for each DIFFERENT operation (usually a UUID)
      2. Reuse that SAME key if retrying that SAME operation
      3. NEVER reuse a key for a genuinely DIFFERENT operation

        # Good: same operation, same key
        POST /payments  Idempotency-Key: pay_abc123  {amount: 100}
        POST /payments  Idempotency-Key: pay_abc123  {amount: 100}  # Retry — fine!

        # Bad: different operation, same key (server should REJECT this)
        POST /payments  Idempotency-Key: pay_abc123  {amount: 200}  # Error!

  6.3 HOW THE SERVER TRACKS EACH KEY (a little state machine)

    Key status    | Server's action
    ----------------|-----------------------------------------------
    Not found        | Process the request; store it as "processing"
    Processing        | Return "409 Conflict" (another one's already running)
    Completed         | Return the CACHED response — don't redo the work
    Failed             | Delete the key, allow a fresh retry

    That "processing" state matters a lot: it stops two SIMULTANEOUS
    identical requests from both slipping through and both executing.

  REAL EXAMPLE: Stripe (the payment company) built idempotency keys
  into the CORE of their entire payments API. If your network drops
  right after Stripe charges the card but before you get the response,
  you just retry with the same key — Stripe recognizes it, skips
  re-charging, and hands you back the original result. Your customer
  never gets double-charged.

  If you're building ANYTHING that touches payments, orders, or
  bookings, idempotency keys should be a DEFAULT design choice, not
  something you bolt on later after a scary incident.


--------------------------------------------------------------------------------
  7. STRATEGY 5 — BLOOM FILTERS (for MASSIVE volumes, approximate answers)
--------------------------------------------------------------------------------

  ANALOGY: Imagine a bouncer who can't possibly memorize every single
  name on a 10-million-person guest list, but has a magic trick: they
  can tell you, almost instantly, "nope, definitely not on the list" OR
  "hmm, might be on the list, let me actually go check the paper" — and
  the "definitely not" answer is NEVER wrong.

  When you're processing MILLIONS of items per second, checking a
  database or Redis for EVERY SINGLE one becomes the bottleneck itself
  (all those network round trips add up). Bloom filters fix this by
  keeping a tiny, in-memory "fuzzy summary" that answers "have I seen
  this before?" in microseconds.

  7.1 HOW A BLOOM FILTER WORKS

    It's just a big array of bits (0s and 1s), plus a few hash functions.

    ADDING AN ITEM (e.g. "msg_123"):

        Item: msg_123
              │
        ┌─────┼─────┬─────────┐
        ▼     ▼      ▼
      Hash1  Hash2  Hash3
        │     │      │
        ▼     ▼      ▼
      pos 2  pos 7  pos 4

      Bit array:  0 0 1 0 1 0 0 1 0 0 0 0
                      ↑   ↑       ↑
                    (positions 2, 4, and 7 got flipped to 1)

    CHECKING AN ITEM:
      1. Run it through the SAME hash functions to get its bit positions
      2. Are ALL of those positions set to 1?
      3. If EVEN ONE bit is 0 → this item was DEFINITELY never added
      4. If ALL bits are 1 → this item was PROBABLY added (there's a
         small chance this is a coincidence from OTHER items — a
         "false positive")

    THE KEY GUARANTEE: a Bloom filter NEVER says "not seen" for
    something it actually HAS seen (no false negatives, ever). But it
    CAN occasionally say "maybe seen" for something brand new (false
    positives are possible, but the RATE is tunable/controllable).

  7.2 USE IT AS A PRE-FILTER, NOT THE FINAL ANSWER

        New Message ID
              │
              ▼
        Check Bloom Filter
              │
       ┌──────┴───────┐
       ▼               ▼
   "Definitely      "Maybe seen
    NEW"             before"
       │               │
       ▼               ▼
   Process it     Check Redis (the real source of truth)
       │               │
       ▼          ┌────┴────┐
  Add to Bloom     ▼         ▼
   + Redis      Found:    Not found:
              Skip, dup   Process it,
                          add to both

    Since MOST items in a typical stream are brand new (not
    duplicates), the Bloom filter says "definitely new" for the vast
    majority, and you can SKIP the expensive Redis/database lookup
    entirely for those. Only the rare "maybe seen" cases need the
    slower, exact check.

  7.3 WHY BOTHER? THE SPACE SAVINGS ARE HUGE

    For 100 million items:

        Storage type              | Size needed
        ----------------------------|---------------
        Exact hash set              | ~3.2 GB
        Bloom filter (1% false pos) | ~120 MB

    That's roughly a 25x reduction in memory — and a Bloom filter can
    live entirely in your app's own memory, with ZERO network trips.

  7.4 WHEN SHOULD YOU ACTUALLY USE ONE?

    Use case                        | Bloom filter OK? | Why
    -----------------------------------|--------------------|--------------------------------
    Web crawler (tracking visited URLs)| Yes                | Worst case, you skip a URL — no big deal
    Spam filter                        | Yes                | An occasional false positive is fine
    "Does this exist in cache?" check  | Yes                | Avoids expensive backend lookups
    Payment deduplication              | NO                 | You need EXACT correctness here
    Message queue (with backup check)  | Yes                | As a pre-filter, backed by an exact store

    Rule of thumb: use Bloom filters when being WRONG occasionally is
    cheap or easily double-checked. Never use them ALONE for anything
    where a mistake costs real money or trust.


--------------------------------------------------------------------------------
  8. STRATEGY 6 — DEDUPLICATION IN EVENT-DRIVEN SYSTEMS (the outbox pattern)
--------------------------------------------------------------------------------

  Here's a sneaky problem: a service often needs to (a) save something
  to its OWN database AND (b) tell OTHER services about it by
  publishing an "event" to a message broker. These are TWO separate
  systems that DON'T share a single all-or-nothing transaction. That
  creates a gap where things can go wrong:

    SCENARIO 1 — Event LOST:
        Create Order → Commit to DB → [CRASH before publishing] →
        Event never gets sent → other services never find out 😱

    SCENARIO 2 — Event DUPLICATED:
        Create Order → Commit to DB → Publish Event →
        [crash before marking "sent"] → on restart, publishes AGAIN

  Both are bad! Lost events break things downstream. Duplicate events
  cause double-processing (back to square one).

  ANALOGY: Imagine mailing an important letter. If you mail it, but
  then can't remember whether you actually dropped it in the mailbox
  or not, you might mail it again "just in case" — and now the
  recipient gets two copies. OR you might assume you mailed it when you
  actually didn't, and it never arrives at all.

  8.1 THE FIX: THE "OUTBOX" PATTERN

    Instead of trying to write to the DATABASE and the QUEUE at the
    same instant (which can't be made perfectly atomic across two
    separate systems), you write the event into a special "outbox"
    TABLE, inside the SAME database transaction as your actual data:

        [SAME TRANSACTION]
          Insert Order row
          Insert Event row into "Outbox" table
                    │
                    ▼
        [SEPARATE background process]
          Poll the Outbox table
                    │
                    ▼
          Publish to the message Queue
                    │
                    ▼
          Mark that outbox row as "published"
                    │
                    ▼
        [Consumer, elsewhere]
          Deduplicate using the event's ID before processing

    Because the Order row and the Outbox row are written in ONE
    database transaction, they either BOTH get saved, or NEITHER does
    — no more "saved the order but forgot to send the event" problem.

    BUT — there's still a catch: the background publisher itself might
    crash AFTER successfully publishing but BEFORE marking the row as
    "sent." On restart, it'll publish that SAME event again. So the
    outbox pattern doesn't eliminate duplicates — it just trades "lost
    events" for "occasional duplicate events," which is a MUCH safer
    trade because...

  8.2 ...CONSUMERS STILL NEED TO DEDUPLICATE

    The outbox pattern only guarantees "at-least-once" event delivery,
    NOT "exactly-once." That means duplicates are still an EXPECTED,
    NORMAL part of the system — not some rare edge case.

    The fix on the consumer side is exactly what we covered in Section
    4: track which event IDs you've already processed, make your
    processing logic idempotent, and/or use database unique constraints
    so a duplicate insert just gets rejected automatically.


--------------------------------------------------------------------------------
  9. SIDE-BY-SIDE COMPARISON OF ALL 6 STRATEGIES
--------------------------------------------------------------------------------

  Strategy               | Space cost      | Speed             | Accuracy    | Best for
  --------------------------|-------------------|---------------------|---------------|---------------------------
  Content Hashing           | 1 hash per file   | Instant lookup       | Exact         | File storage / uploads
  Chunk Deduplication       | 1 record/chunk    | Instant per chunk    | Exact         | Large files, backups
  Message ID Tracking       | 1 record/message  | Instant lookup       | Exact         | Message queues
  DB Unique Constraints     | Small index cost  | Fast (log n)         | Exact         | Database record dedup
  Idempotency Keys          | 1 key per request | Instant lookup       | Exact         | Payments, order APIs
  Bloom Filters             | TINY (bits only)  | A few hash lookups   | Approximate!  | Very high-volume streams

  Simple way to remember it:
    - Need PERFECT accuracy (money, orders)? → Exact methods (hashing,
      unique constraints, idempotency keys)
    - Have MASSIVE volume and can tolerate occasional double-checks?
      → Bloom filter as a cheap pre-filter, backed by an exact store


--------------------------------------------------------------------------------
  10. DECISION FLOWCHART — "What am I actually deduplicating?"
--------------------------------------------------------------------------------

                      What are you deduplicating?
                                │
        ┌───────────┬──────────┼──────────────┬──────────────┐
        ▼           ▼          ▼              ▼
    Files/Blobs   Messages   DB Records    API Requests
        │           │          │              │
        ▼           ▼          ▼              ▼
    How big?    How much     Use UNIQUE    Use IDEMPOTENCY
        │        volume?     CONSTRAINTS      KEYS
    ┌───┴───┐       │        (+ upserts)
    ▼       ▼    ┌───┴────┐
  Small   Large  ▼        ▼
  (<10MB) (>10MB) <1M/day  >1M/day
    │       │       │        │
    ▼       ▼       ▼        ▼
  Content  Chunk  Message   Bloom Filter
  Hash     Dedup  ID+Redis  + Exact Store
                  (exact
                   store)


--------------------------------------------------------------------------------
  11. A FEW BEST PRACTICES WORTH REMEMBERING
--------------------------------------------------------------------------------

  1) PICK THE RIGHT "ZOOM LEVEL"
     Whole-file dedup misses near-duplicate files. Byte-by-byte dedup
     has so much overhead it can actually cost MORE than it saves.
     For files, chunks around 4KB-64KB are usually the sweet spot.

  2) SET SENSIBLE EXPIRATION TIMES (TTLs)
     Your "have I seen this before?" store can't grow forever, or it'll
     eventually run out of space.

        Dedup type        | Suggested TTL   | Why
        --------------------|-------------------|--------------------------------
        Message dedup        | 24 hours          | Covers most realistic retry windows
        Idempotency keys      | 24 hours          | Covers a typical user session
        Content hashes         | No expiry          | Use reference counting instead
                                                     (delete only when NOTHING points to it)

  3) TREAT DIFFERENT FAILURES DIFFERENTLY
     Don't mark something as "done" if it actually failed! But also
     don't retry forever if it's a PERMANENT failure (like bad data)
     — that just wastes resources and trains your team to ignore alerts.

        Error type                     | What to do
        ----------------------------------|-----------------------------------------
        Temporary (timeout, network blip) | DON'T mark as processed — allow a retry
        Permanent (bad data, validation)   | DO mark as processed — stop retrying junk
        Unknown / unclear                  | Log it and alert a human

  4) ACTUALLY MEASURE IT
     Track things like: what % of items turn out to be duplicates, how
     much storage you're saving, how fast your dedup checks run, and
     (for Bloom filters specifically) how often a "maybe duplicate"
     turns out to actually be brand new. A surprisingly LOW duplicate
     rate could mean things are working great... or it could mean your
     deduplication is silently broken. Context matters — cross-check
     against your retry/error rates.


--------------------------------------------------------------------------------
  12. QUICK-REFERENCE DECISION CHECKLIST
--------------------------------------------------------------------------------

  Before finalizing your approach, ask:

    [ ] What TYPE of duplicate am I actually dealing with — a file, a
        message, a database record, or an API request? (Each needs a
        different tool — don't reach for the wrong one.)
    [ ] Does this operation involve REAL money or irreversible
        consequences (like payments)? → Exact methods only, no Bloom filters.
    [ ] Am I handling the "check-then-act" race condition properly, or
        did I accidentally leave a gap where two things can slip through
        at once? (Use atomic operations: SET-if-not-exists, DB upserts.)
    [ ] Have I set a sensible TTL/expiry for my dedup tracking data, so
        it doesn't grow forever?
    [ ] What happens if a worker CRASHES halfway through? Does my
        design recover gracefully (via TTL expiry) instead of getting
        permanently "stuck"?
    [ ] For fuzzy/messy data — do I have a plan for LOW-CONFIDENCE
        matches, or am I risking incorrectly merging two different
        records together?
    [ ] If I'm using an event-driven system with a database + a message
        queue, am I using the outbox pattern instead of trying to write
        to both at once?
    [ ] Am I ALSO deduplicating on the consumer side, even if I already
        used an outbox pattern on the producer side? (You still need both!)
    [ ] Is my volume high enough that I actually NEED a Bloom filter
        pre-filter, or would a simple exact check be totally fine (and
        simpler to reason about)?


--------------------------------------------------------------------------------
  TL;DR — THE ONE THING TO REMEMBER
--------------------------------------------------------------------------------

  You cannot design duplicates out of a distributed system — networks
  drop things, services retry, and people double-click buttons. That's
  just how it is.

  So the real skill isn't PREVENTING duplicates. It's making sure that
  whenever the SAME logical thing happens twice (or ten times), the
  RESULT is still correct — whether that's "store the file once,"
  "process the message once," "one row per email," or "charge the card
  exactly once." Pick the deduplication tool that matches WHERE the
  duplicate can occur, and design for retries to be safe by default.

================================================================================
*/